# 01 — Análisis Exploratorio: Call Center

**Objetivo:** explorar las métricas operativas del call center para entender la distribución de variables,
identificar correlaciones clave y segmentar registros por nivel de servicio.

**Dataset:** 1,251 registros periódicos con 8 métricas de rendimiento.

**Hallazgos principales:**
- Answer Rate promedio: ~92.7% (saludable)
- Service Level promedio: ~70.9% — por debajo del estándar de industria del 80%
- Correlación fuerte entre Waiting Time y Abandoned Calls (r ≈ 0.72)
- Correlación negativa moderada entre Incoming Calls y Service Level (r ≈ -0.49)
- ~18% de registros son "críticos" (Service Level < 50%)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / "src"))

from call_center.data_loader import load_clean, CLEAN_COLS

sns.set_theme(style="whitegrid", palette="muted")
FIGURES = Path.cwd().parent / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
df = load_clean()
print(f"Shape: {df.shape}")
df.head()

## 1. Vista general del dataset

In [ ]:
df.describe().round(3)

In [ ]:
# Nulos
print("Nulos por columna:")
print(df.isnull().sum())

## 2. KPIs del call center

In [ ]:
kpis = {
    "Answer Rate (avg)": f"{df[CLEAN_COLS['answer_rate']].mean():.1%}",
    "Service Level 20s (avg)": f"{df[CLEAN_COLS['service_level']].mean():.1%}",
    "Answer Speed (avg)": f"{df[CLEAN_COLS['answer_speed']].mean():.0f}s",
    "Talk Duration (avg)": f"{df[CLEAN_COLS['talk_duration']].mean():.0f}s ({df[CLEAN_COLS['talk_duration']].mean()/60:.1f} min)",
    "Waiting Time (avg)": f"{df[CLEAN_COLS['waiting_time']].mean():.0f}s ({df[CLEAN_COLS['waiting_time']].mean()/60:.1f} min)",
    "Abandoned Calls (avg)": f"{df[CLEAN_COLS['abandoned']].mean():.1f}",
}

print("── KPIs ──────────────────────────────")
for k, v in kpis.items():
    print(f"  {k:<30} {v}")
print(f"  {'Estándar industria Service Level':<30} 80.0%  ← por encima del promedio actual")

## 3. Distribuciones

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

col_labels = {
    CLEAN_COLS["incoming"]:       "Incoming Calls",
    CLEAN_COLS["answered"]:       "Answered Calls",
    CLEAN_COLS["answer_rate"]:    "Answer Rate",
    CLEAN_COLS["abandoned"]:      "Abandoned Calls",
    CLEAN_COLS["answer_speed"]:   "Answer Speed (s)",
    CLEAN_COLS["talk_duration"]:  "Talk Duration (s)",
    CLEAN_COLS["waiting_time"]:   "Waiting Time (s)",
    CLEAN_COLS["service_level"]:  "Service Level 20s",
}

for ax, (col, label) in zip(axes, col_labels.items()):
    sns.histplot(df[col], ax=ax, kde=True, bins=30)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("")

fig.suptitle("Distribuciones — Call Center Dataset", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / "01_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Boxplots para detectar outliers
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

time_cols = [
    (CLEAN_COLS["answer_speed"], "Answer Speed (s)"),
    (CLEAN_COLS["talk_duration"], "Talk Duration (s)"),
    (CLEAN_COLS["waiting_time"], "Waiting Time (s)"),
]

for ax, (col, label) in zip(axes, time_cols):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(label)

fig.suptitle("Distribución de tiempos — outliers", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / "02_time_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Análisis de correlaciones

In [ ]:
corr = df.corr(numeric_only=True).round(2)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    ax=ax,
    linewidths=0.5,
)
ax.set_title("Matriz de correlaciones (Pearson)", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / "03_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# Correlaciones clave
print("\nCorrelaciones clave:")
print(f"  Waiting Time ↔ Abandoned Calls:  r = {corr.loc[CLEAN_COLS['waiting_time'], CLEAN_COLS['abandoned']]:.2f}")
print(f"  Incoming Calls ↔ Service Level:   r = {corr.loc[CLEAN_COLS['incoming'], CLEAN_COLS['service_level']]:.2f}")

In [ ]:
# Scatter: Waiting Time vs Abandoned Calls
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(
    data=df,
    x=CLEAN_COLS["waiting_time"],
    y=CLEAN_COLS["abandoned"],
    alpha=0.4,
    ax=axes[0],
)
axes[0].set_title("Waiting Time vs Abandoned Calls")
axes[0].set_xlabel("Waiting Time (s)")
axes[0].set_ylabel("Abandoned Calls")

sns.scatterplot(
    data=df,
    x=CLEAN_COLS["incoming"],
    y=CLEAN_COLS["service_level"],
    alpha=0.4,
    ax=axes[1],
)
axes[1].set_title("Incoming Calls vs Service Level")
axes[1].set_xlabel("Incoming Calls")
axes[1].set_ylabel("Service Level 20s")
axes[1].axhline(0.80, color="red", linestyle="--", label="Estándar 80%")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / "04_key_scatterplots.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Segmentación de registros

Clasificamos los periodos por nivel de servicio para identificar qué fracción del tiempo el call center opera por debajo del estándar.

In [ ]:
def segment_service_level(sl: float) -> str:
    if sl >= 0.80:
        return "Bueno (≥80%)"
    elif sl >= 0.50:
        return "Regular (50-80%)"
    else:
        return "Crítico (<50%)"

df["segment"] = df[CLEAN_COLS["service_level"]].apply(segment_service_level)

counts = df["segment"].value_counts()
pcts = (counts / len(df) * 100).round(1)

print("Segmentación por Service Level:")
for seg in ["Bueno (≥80%)", "Regular (50-80%)", "Crítico (<50%)"]:
    print(f"  {seg:<22} {counts.get(seg, 0):>4} registros  ({pcts.get(seg, 0):.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
order = ["Bueno (≥80%)", "Regular (50-80%)", "Crítico (<50%)"]
colors = ["#4CAF50", "#FFC107", "#F44336"]
sizes = [counts.get(s, 0) for s in order]

axes[0].pie(sizes, labels=order, colors=colors, autopct="%1.1f%%", startangle=140)
axes[0].set_title("Distribución por segmento de servicio")

# KPIs por segmento
seg_stats = df.groupby("segment")[[CLEAN_COLS["waiting_time"], CLEAN_COLS["abandoned"]]].mean().round(1)
seg_stats.columns = ["Waiting Time (s)", "Abandoned Calls"]
seg_stats.loc[order].plot(kind="bar", ax=axes[1], color=["#5C85D6", "#E07B54"])
axes[1].set_title("Waiting Time y Abandonos promedio por segmento")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / "05_segments.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Limpiar columna auxiliar
df = df.drop(columns=["segment"])

## 6. Resumen de hallazgos

| Métrica | Valor | Referencia | Estado |
|---|---|---|---|
| Answer Rate (avg) | ~92.7% | >90% aceptable | ✅ |
| Service Level 20s (avg) | ~70.9% | >80% estándar | ⚠️ |
| Answer Speed (avg) | ~27s | <20s ideal | ⚠️ |
| Waiting Time (avg) | ~366s (6.1 min) | — | — |
| Correlación Wait ↔ Abandoned | r ≈ 0.72 | — | Fuerte positiva |
| Correlación Incoming ↔ SL | r ≈ -0.49 | — | Moderada negativa |
| Registros críticos (<50% SL) | ~18% | — | — |